# HW: Multimodal 2D Optical–SAR Image Registration (OSdataset)

## Deep Learning / Computer Vision — Programming Assignment

**Big picture.** Optical and SAR (radar) sensors image the same scene very
differently. You will (1) apply a *known* affine transform to SAR images to create
controlled misalignment, then (2) **recover** that transform two ways — an
intensity-based method (Mutual Information) and a feature-based method
(ORB + RANSAC) — and (3) **measure** how well each recovered it.

**Notebook input →** paired `opt_*.png` / `sar_*.png` images (256×256):
https://www.kaggle.com/datasets/hemanthpadala/osdataset
**Notebook output →** a synthetic misaligned SAR set with ground-truth params,
recovered transforms, aligned images, and error numbers.

You are expected to **read the PIL / OpenCV / SimpleITK docs and work out the
details yourself.** Cells state only the input/output of each function.

### Deliverables & grading (100 pts)
| Part | Topic | Pts |
|------|-------|-----|
| 1 | Explore & visualize pairs | 10 |
| 2 | Affine matrices from parameters | 20 |
| 3 | Synthetic misaligned SAR set | 15 |
| 4 | Intensity-based registration (MI) | 20 |
| 5 | Feature-based registration (ORB) | 20 |
| 6 | Evaluation & comparison | 10 |
| — | Written answers (✍️) | 5 |

> Fill in every `# TODO` and ✍️ question. Keep the provided asserts and headers.


## Part 0 — Setup

In [ ]:
# !pip install opencv-python-headless scikit-image pillow numpy matplotlib


In [ ]:
import os, glob, math, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
random.seed(0); np.random.seed(0)


## Part 1 — Explore the dataset  *(10 pts)*

In [ ]:
# TODO: set DIR_PATH to the test folder; collect sorted opt_files and sar_files.
DIR_PATH = "/kaggle/input/datasets/hemanthpadala/osdataset/OSdataset/256/test"
opt_files = sorted(glob.glob(os.path.join(DIR_PATH, "opt*.png")))
sar_files = sorted(glob.glob(os.path.join(DIR_PATH, "sar*.png")))

assert DIR_PATH and os.path.isdir(DIR_PATH), "DIR_PATH not found"
assert opt_files and sar_files, "No opt/sar images found"
print(f"{len(opt_files)} optical, {len(sar_files)} SAR")


In [ ]:
# TODO: show_pair(index) -> displays the optical and SAR image at `index` side by
# side. Call it for indices 0 and 1.
def show_pair(index):
    opt = plt.imread(opt_files[index])
    sar = plt.imread(sar_files[index])

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(opt, cmap="gray")
    axes[0].set_title(f"Optical [{index}]: {os.path.basename(opt_files[index])}")
    axes[1].imshow(sar, cmap="gray")
    axes[1].set_title(f"SAR [{index}]: {os.path.basename(sar_files[index])}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_pair(0)
show_pair(1)


✍️ **Q1.1** Give two visual differences between optical and SAR images, and
why they make sum-of-squared-differences registration unreliable here.

**Your answer:** _

SAR images have grainy, high-frequency multiplicative speckle noise that optical images don't have, even over uniform surfaces.

A surface that's bright in optical can appear dark in SAR, while rough or man-made structures (buildings) can appear very bright in SAR due to corner-reflector effects, even if they look ordinary in optical.

SSD assumes that corresponding structures have **similar intensity** in both images. Since brightness relationships between optical and SAR are unrelated for many surfaces, and SAR has extra speckle noise, minimizing SSD would not correspond to correct alignment. It would instead be dominated by these intensity mismatches, not true geometric offset.

## Part 2 — Affine transforms from parameters  *(20 pts)*

In [ ]:
def get_affine_matrix(shift_x, shift_y, rotation_deg, scale_x, scale_y, shear_deg):
    """Build a 2D affine from interpretable parameters.

    Input : translation (px), rotation (deg), scale (x,y), shear (deg).
    Output: the 6 values (a,b,c,d,e,f) = first two rows of the 3x3 affine,
            in the layout PIL's Image.transform(..., Image.AFFINE, data=...) expects.
    """
    w, h = (256, 256)
    cx, cy = w / 2.0, h / 2.0

    theta = np.deg2rad(rotation_deg)
    shear = np.deg2rad(shear_deg)

    S = np.array([[scale_x, 0],[0, scale_y]])
    Sh = np.array([[1, np.tan(shear)],[0, 1]])
    R = np.array([[np.cos(theta), -np.sin(theta)],[np.sin(theta),  np.cos(theta)]])

    M_linear = R @ Sh @ S

    center = np.array([cx, cy])
    t = np.array([shift_x, shift_y]) + center - M_linear @ center

    forward = np.eye(3)
    forward[:2, :2] = M_linear
    forward[:2, 2] = t

    inverse = np.linalg.inv(forward)

    a, b, c = inverse[0, 0], inverse[0, 1], inverse[0, 2]
    d, e, f = inverse[1, 0], inverse[1, 1], inverse[1, 2]
    return (a, b, c, d, e, f)

_id = np.round(get_affine_matrix(0, 0, 0, 1.0, 1.0, 0), 6)
assert np.allclose(_id, [1, 0, 0, 0, 1, 0]), f"identity check failed: {_id}"
print("identity ok:", _id)


In [ ]:
# TODO: apply_affine(img, params) -> transformed PIL image, where
# params = (shift_x, shift_y, rotation_deg, scale_x, scale_y, shear_deg).
# Then demo: original vs a +30deg rotation vs a (20,0) shift.
def apply_affine(img, params):
    coeffs = get_affine_matrix(*params)
    return img.transform(img.size, Image.AFFINE, coeffs, resample=Image.BILINEAR)

img0 = Image.open(opt_files[0]).convert("L")
img_rot30  = apply_affine(img0, (0, 0, 30, 1.0, 1.0, 0))
img_shift  = apply_affine(img0, (20, 0, 0, 1.0, 1.0, 0))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img0, cmap="gray");      axes[0].set_title("Original")
axes[1].imshow(img_rot30, cmap="gray"); axes[1].set_title("+30deg rotation")
axes[2].imshow(img_shift, cmap="gray"); axes[2].set_title("(20,0) shift")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


✍️ **Q2.1** Does multiplying the component matrices in a different order give
the same transform? Why does order matter?

✍️ **Q2.2** PIL's `Image.AFFINE` maps *output* pixels back to *input* pixels
(inverse mapping). What does that imply when you later compare your estimated
transform to the true one?

**Your answers:**

- Q2.1: _No, multiplying the component matrices in a different order generally gives a **different transform**, because matrix multiplication is not commutative. For example, rotating then shearing moves points along axes that have already been rotated, while shearing then rotating shears along the original axes before rotation. Order only doesn't matter for cases like uniform scaling combined with rotation._
  
- Q2.2: _Because PIL's `Image.AFFINE` uses inverse (backward) mapping, the 6 coefficients we compute and pass in already represent the **inverse** of the transform we want to apply (this is also why `get_affine_matrix` inverts the forward matrix before returning it). This matters for evaluation: true transform describing how we displaced the SAR image, and the transform our registration method estimates, need to be expressed in the **same direction** (both forward, or both inverse) before comparing parameters directly. Comparing a forward transform to an inverse transform will look like a large error even if the registration is actually correct._



## Part 3 — Synthetic misaligned SAR set  *(15 pts)*

In [ ]:
OUTPUT_DIR = "/kaggle/working/modified_SAR_test/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_SHIFT, MAX_ROT, MAX_SHEAR = 15, 30, 10
SCALE_MIN, SCALE_MAX = 0.85, 1.15


In [ ]:
# TODO: random_params() -> params sampled within the ranges above.
def random_params():
    shift_x = np.random.uniform(-MAX_SHIFT, MAX_SHIFT)
    shift_y = np.random.uniform(-MAX_SHIFT, MAX_SHIFT)
    rotation_deg = np.random.uniform(-MAX_ROT, MAX_ROT)
    scale_x = np.random.uniform(SCALE_MIN, SCALE_MAX)
    scale_y = np.random.uniform(SCALE_MIN, SCALE_MAX)
    shear_deg = np.random.uniform(-MAX_SHEAR, MAX_SHEAR)
    return (shift_x, shift_y, rotation_deg, scale_x, scale_y, shear_deg)

# TODO: make_variant(sar_img, basename, idx) -> applies a random affine, saves the
# result to OUTPUT_DIR (encode the params in the filename), and RETURNS the exact
# params used.
def make_variant(sar_img, basename, idx):
    params = random_params()
    shift_x, shift_y, rotation_deg, scale_x, scale_y, shear_deg = params

    warped = apply_affine(sar_img, params)

    fname = (
        f"{basename}_v{idx}"
        f"_sx{shift_x:.2f}_sy{shift_y:.2f}"
        f"_rot{rotation_deg:.2f}"
        f"_scx{scale_x:.3f}_scy{scale_y:.3f}"
        f"_sh{shear_deg:.2f}.png"
    )
    out_path = os.path.join(OUTPUT_DIR, fname)
    warped.save(out_path)

    return params


In [ ]:
# TODO: for the first ~20 SAR files, make num_variants=3 each; store
# truth[variant_filename] = params_used.
truth = {}
num_variants = 3
SUBSET = sar_files[:20]
for sar_path in SUBSET:
    basename = os.path.splitext(os.path.basename(sar_path))[0]
    sar_img = Image.open(sar_path).convert("L")

    for idx in range(num_variants):
        params = make_variant(sar_img, basename, idx)
        variant_fname = (
            f"{basename}_v{idx}"
            f"_sx{params[0]:.2f}_sy{params[1]:.2f}"
            f"_rot{params[2]:.2f}"
            f"_scx{params[3]:.3f}_scy{params[4]:.3f}"
            f"_sh{params[5]:.2f}.png"
        )
        truth[variant_fname] = params

print(f"{len(truth)} misaligned SAR images in {OUTPUT_DIR}")
assert len(truth) > 0


## Part 4 — Intensity-based registration (Mutual Information)  *(20 pts)*

Align a misaligned SAR (moving) to its optical (fixed) by maximizing Mutual
Information. Use SimpleITK (recommended) or another library.

In [ ]:
import SimpleITK as sitk

def to_sitk(pil_gray):
    """PIL grayscale -> 2D sitk.Image (Float32)."""
    return sitk.GetImageFromArray(np.asarray(pil_gray, dtype=np.float32))

# TODO: register_mi(fixed_pil, moving_pil) -> sitk transform aligning moving->fixed
# (multimodal: use a Mutual Information metric).
def register_mi(fixed_pil, moving_pil, angle_range=45, angle_step=15):
    fixed = to_sitk(fixed_pil)
    moving = to_sitk(moving_pil)

    def run(initial_transform, n_iter):
        R = sitk.ImageRegistrationMethod()
        R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
        R.SetMetricSamplingStrategy(R.RANDOM)
        R.SetMetricSamplingPercentage(0.5)
        R.SetInterpolator(sitk.sitkLinear)
        R.SetOptimizerAsRegularStepGradientDescent(
            learningRate=2.0, minStep=1e-5, numberOfIterations=n_iter, relaxationFactor=0.5
        )
        R.SetOptimizerScalesFromPhysicalShift()
        R.SetShrinkFactorsPerLevel([4, 2, 1])
        R.SetSmoothingSigmasPerLevel([2, 1, 0])
        R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        R.SetInitialTransform(initial_transform, inPlace=False)
        transform = R.Execute(fixed, moving)
        if isinstance(transform, sitk.CompositeTransform):
            transform = transform.GetNthTransform(0)
        return transform, R.GetMetricValue()

    best_transform, best_mi = None, float("inf")
    for angle_deg in range(-angle_range, angle_range + 1, angle_step):
        init = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.Euler2DTransform(),
            sitk.CenteredTransformInitializerFilter.MOMENTS
        )
        init.SetAngle(np.deg2rad(angle_deg))
        transform, mi = run(init, n_iter=100)
        if mi < best_mi:
            best_transform, best_mi = transform, mi

    affine_init = sitk.AffineTransform(2)
    affine_init.SetCenter(best_transform.GetCenter())
    affine_init.SetMatrix(best_transform.GetMatrix())
    affine_init.SetTranslation(best_transform.GetTranslation())

    final_transform, _ = run(affine_init, n_iter=300)
    return final_transform

def apply_sitk_transform(fixed_pil, moving_pil, transform):
    """Resample moving into fixed's grid; return a PIL image."""
    out = sitk.Resample(to_sitk(moving_pil), to_sitk(fixed_pil), transform,
                        sitk.sitkLinear, 0.0, sitk.sitkFloat32)
    return Image.fromarray(np.clip(sitk.GetArrayFromImage(out), 0, 255).astype(np.uint8))


In [ ]:
# TODO: pick one variant from `truth`, find its matching optical image, run
variant_fname = list(truth.keys())[1]
variant_params = truth[variant_fname]

basename = variant_fname.split("_v")[0]      
opt_basename = basename.replace("sar", "opt", 1)
opt_path = os.path.join(DIR_PATH, f"{opt_basename}.png")

# register_mi, produce the aligned SAR, and show fixed | misaligned | aligned.
fixed_pil = Image.open(opt_path).convert("L")
moving_pil = Image.open(os.path.join(OUTPUT_DIR, variant_fname)).convert("L")

estimated_tx = register_mi(fixed_pil, moving_pil)
aligned_pil = apply_sitk_transform(fixed_pil, moving_pil, estimated_tx)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(fixed_pil, cmap="gray");   axes[0].set_title("Fixed (optical)")
axes[1].imshow(moving_pil, cmap="gray");  axes[1].set_title("Misaligned SAR")
axes[2].imshow(aligned_pil, cmap="gray"); axes[2].set_title("Aligned SAR (MI)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("True params used:", variant_params)


✍️ **Q4.1** Why does Mutual Information work across optical/SAR when
intensity-difference metrics fail?

**Your answer:** _Mutual Information measures **statistical dependency** between two images' intensity distributions, not their raw intensity similarity. It doesn't require corresponding pixels to have the same brightness but only that knowing one image's intensity at a location tells you something predictable about the other's, even if that relationship is complex or inverted. Intensity-difference metrics like SSD assume similar brightness for matching structures, which fails between optical and SAR since their imaging physics are completely different. MI captures the shared *information* between modalities regardless of how their intensity scales relate._

## Part 5 — Feature-based registration (ORB + RANSAC)  *(20 pts)*

In [ ]:
import cv2

# TODO: register_orb(fixed_pil, moving_pil) -> (M, kp_fixed, kp_moving, matches, inliers)
# where M is the estimated affine/homography aligning moving->fixed (ORB features,
# descriptor matching, RANSAC fit).

def preprocess_for_orb(pil_img, is_sar=False):
    arr = np.array(pil_img).astype(np.uint8)
    if is_sar:
        arr = cv2.medianBlur(arr, 5)

    arr = cv2.equalizeHist(arr)
    edges = cv2.Canny(arr, 50, 150)

    edges = cv2.dilate(edges, np.ones((2, 2), np.uint8), iterations=1)
    return edges
    
def register_orb(fixed_pil, moving_pil, max_features=3000, ratio_thresh=0.85):
    fixed_edges = preprocess_for_orb(fixed_pil, is_sar=False)
    moving_edges = preprocess_for_orb(moving_pil, is_sar=True)

    orb = cv2.ORB_create(nfeatures=max_features, scaleFactor=1.1, nlevels=12)
    kp_fixed, des_fixed = orb.detectAndCompute(fixed_edges, None)
    kp_moving, des_moving = orb.detectAndCompute(moving_edges, None)

    if des_fixed is None or des_moving is None or len(kp_fixed) < 4 or len(kp_moving) < 4:
        return None, kp_fixed, kp_moving, [], []

    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    knn_matches = bf.knnMatch(des_moving, des_fixed, k=2)

    good_matches = []
    for pair in knn_matches:
        if len(pair) == 2:
            m, n = pair
            if m.distance < ratio_thresh * n.distance:
                good_matches.append(m)

    if len(good_matches) < 4:
        return None, kp_fixed, kp_moving, good_matches, []

    src_pts = np.float32([kp_moving[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_fixed[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    M, inlier_mask = cv2.estimateAffine2D(
        src_pts, dst_pts, method=cv2.RANSAC,
        ransacReprojThreshold=4.0, maxIters=10000, confidence=0.995
    )
    if M is None:
        return None, kp_fixed, kp_moving, good_matches, []

    inliers = [good_matches[i] for i in range(len(good_matches)) if inlier_mask[i]]
    return M, kp_fixed, kp_moving, good_matches, inliers



In [ ]:
# TODO: run register_orb on the SAME pair as Part 4; warp the SAR with M, draw the
# inlier matches, and show fixed | misaligned | ORB-aligned. If ORB fails (few
# inliers), report that — it is an expected outcome here.

M, kp_fixed, kp_moving, matches, inliers = register_orb(fixed_pil, moving_pil)
print(f"Total good matches kept: {len(matches)}")
print(f"RANSAC inliers: {len(inliers)}")

if M is None or len(inliers) < 4:
    print("ORB registration failed")
else:
    moving_arr = np.array(moving_pil)
    orb_aligned_arr = cv2.warpAffine(moving_arr, M, fixed_pil.size, flags=cv2.INTER_LINEAR)
    orb_aligned_pil = Image.fromarray(orb_aligned_arr)

    match_img = cv2.drawMatches(
        moving_arr, kp_moving, np.array(fixed_pil), kp_fixed, inliers, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    plt.figure(figsize=(10, 5))
    plt.imshow(match_img, cmap="gray")
    plt.title(f"ORB inlier matches ({len(inliers)})")
    plt.axis("off")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(fixed_pil, cmap="gray");      axes[0].set_title("Fixed (optical)")
    axes[1].imshow(moving_pil, cmap="gray");     axes[1].set_title("Misaligned SAR")
    axes[2].imshow(orb_aligned_pil, cmap="gray"); axes[2].set_title("ORB-aligned SAR")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()



✍️ **Q5.1** How many inliers did RANSAC keep, and was ORB better or worse
than the MI result? Give one reason cross-modal feature matching is hard.

**Your answer:** _Cross-modal feature matching is hard because ORB descriptors rely on locally consistent intensity patterns, but optical and SAR images represent the same location with very different local appearance. SAR's speckle noise and different backscatter behavior mean the same physical spot rarely produces similar-looking keypoints in both modalities. As a result, very few true correspondences survive matching, leaving RANSAC too little reliable data to fit a meaningful transform._

## Part 6 — Evaluation & comparison  *(10 pts)*

In [ ]:
# TODO: checkerboard(a, b, tiles=8) -> np.ndarray interleaving two same-size
# grayscale arrays. Use it for before/after on both the MI and ORB results.
def checkerboard(a, b, tiles=8):
    h, w = a.shape
    out = np.zeros_like(a)
    th, tw = h // tiles, w // tiles
    for i in range(tiles):
        for j in range(tiles):
            r0, r1 = i * th, (i + 1) * th if i < tiles - 1 else h
            c0, c1 = j * tw, (j + 1) * tw if j < tiles - 1 else w
            out[r0:r1, c0:c1] = a[r0:r1, c0:c1] if (i + j) % 2 == 0 else b[r0:r1, c0:c1]
    return out


In [ ]:
# TODO: corner_error(H_true, H_est, size=256) -> mean pixel distance between the
# four image corners mapped by the true vs the estimated transform. Report it for
# the MI and ORB transforms and state which won.
def corner_error(H_true, H_est, size=256):
    corners = np.array([
        [0, 0, 1],
        [size - 1, 0, 1],
        [0, size - 1, 1],
        [size - 1, size - 1, 1],
    ]).T  

    pts_true = H_true @ corners  
    pts_est = H_est @ corners  

    dists = np.linalg.norm(pts_true - pts_est, axis=0)
    return dists.mean()

def invert_params_to_transform(params, image_size=(256, 256)):

    a, b, c, d, e, f = get_affine_matrix(*params)
    INV = np.array([[a, b, c],
                     [d, e, f],
                     [0, 0, 1]])
    FWD = np.linalg.inv(INV)

    M = FWD[:2, :2]
    t = FWD[:2, 2]

    tx = sitk.AffineTransform(2)
    tx.SetMatrix(M.flatten())
    tx.SetTranslation(t)
    return tx

def sitk_to_forward_matrix(tx):
    
    inv = tx.GetInverse() 
    A = np.array(inv.GetMatrix()).reshape(2, 2)
    c = np.array(inv.GetCenter())
    t = np.array(inv.GetTranslation())
    t_full = t + c - A @ c
    return np.hstack([A, t_full.reshape(2, 1)])

true_tx_sitk = invert_params_to_transform(variant_params, image_size=fixed_pil.size)
H_true = sitk_to_forward_matrix(true_tx_sitk)

H_mi = sitk_to_forward_matrix(estimated_tx)

H_orb = M if M is not None else None

err_mi = corner_error(H_true, H_mi, size=fixed_pil.size[0])
print(f"MI corner error:  {err_mi:.2f} px")

if H_orb is not None:
    err_orb = corner_error(H_true, H_orb, size=fixed_pil.size[0])
    print(f"ORB corner error: {err_orb:.2f} px")
else:
    print("registration failed")

def normalize(img):
    lo, hi = np.percentile(img, [1, 99])
    return np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)

fixed_n = normalize(np.array(fixed_pil).astype(np.float32))
mi_n    = normalize(np.array(aligned_pil).astype(np.float32))
orb_n   = normalize(np.array(orb_aligned_pil).astype(np.float32)) if H_orb is not None else None
moving_n = normalize(np.array(moving_pil).astype(np.float32))

cb_before   = checkerboard(fixed_n, moving_n, tiles=8)
cb_after_mi = checkerboard(fixed_n, mi_n, tiles=8)

fig, axes = plt.subplots(1, 3 if orb_n is None else 3, figsize=(15, 5))
axes[0].imshow(cb_before, cmap="gray");   axes[0].set_title("Before")
axes[1].imshow(cb_after_mi, cmap="gray"); axes[1].set_title("After (MI)")
if orb_n is not None:
    cb_after_orb = checkerboard(fixed_n, orb_n, tiles=8)
    axes[2].imshow(cb_after_orb, cmap="gray"); axes[2].set_title("After (ORB)")
else:
    axes[2].axis("off")
    axes[2].set_title("ORB failed")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

✍️ **Q6.1 (Reflection)** Compare MI vs ORB+RANSAC (accuracy, speed,
robustness) for optical↔SAR. Which would you deploy, and why?

### Bonus (+10)
- Batch corner-error histogram per method over all variants.
- A deep multimodal descriptor vs ORB.
- Coarse-to-fine MI: speed/accuracy trade-off.

**Your answer / bonus notes:** _For optical to SAR registration on this pair, **Mutual Information clearly won on accuracy**: corner error of ~21px versus ORB+RANSAC's ~276px, with the checkerboard visualization confirming MI produced a structurally coherent alignment while ORB collapsed the image into a degenerate sheared shape.

**Accuracy:** MI wins decisively. It doesn't depend on finding visually repeatable keypoints across modalities — it only needs a statistical relationship between intensity distributions, which holds even when SAR speckle and backscatter physics make the same location look completely different from its optical counterpart. ORB's local descriptors simply don't transfer across this modality gap; only ~7% of its candidate matches were even geometrically consistent, and those few were likely coincidental rather than true correspondences.

**Speed:** ORB+RANSAC is much faster in principle — feature detection and matching are near-instant, versus MI's iterative optimization. However, MI's slower runtime bought a working result, while ORB's speed was irrelevant since it failed to converge to anything usable.

**Robustness:** MI is more robust to modality differences by design, but it does require reasonable initialization and can still converge to a wrong basin without care. ORB's failure mode is worse for deployment risk: it can silently return a *confident-looking* transform that is actually garbage, which is dangerous if not visually or quantitatively double-checked.

**Deployment choice: Mutual Information.** For real optical-SAR alignment, a wrong-but-confident transform (ORB's failure mode here) is more dangerous than a slower correct one. I'd deploy an MI-based pipeline with a coarse-to-fine strategy to mitigate its main weakness (sensitivity to initialization for large rotations). ORB could still be useful as a fast, cheap first-pass initializer in easier cases, but shouldn't be trusted as the final registration method for this modality pair._

In [ ]:
#Coarse-to-fine MI: speed/accuracy trade-off.
import time

def register_mi_with_schedule(fixed_pil, moving_pil, shrink_factors, sigmas, angle_range=45, angle_step=15, n_iter=100, refine_iter=300):

    fixed = to_sitk(fixed_pil)
    moving = to_sitk(moving_pil)

    def run(init_transform, n):
        R = sitk.ImageRegistrationMethod()
        R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
        R.SetMetricSamplingStrategy(R.RANDOM)
        R.SetMetricSamplingPercentage(0.5)
        R.SetInterpolator(sitk.sitkLinear)
        R.SetOptimizerAsRegularStepGradientDescent(
            learningRate=2.0, minStep=1e-5, numberOfIterations=n, relaxationFactor=0.5
        )
        R.SetOptimizerScalesFromPhysicalShift()
        R.SetShrinkFactorsPerLevel(shrink_factors)
        R.SetSmoothingSigmasPerLevel(sigmas)
        R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        R.SetInitialTransform(init_transform, inPlace=False)
        tx = R.Execute(fixed, moving)
        if isinstance(tx, sitk.CompositeTransform):
            tx = tx.GetNthTransform(0)
        return tx, R.GetMetricValue()

    best_tx, best_score = None, float("inf")
    for angle_deg in range(-angle_range, angle_range + 1, angle_step):
        init = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.Euler2DTransform(),
            sitk.CenteredTransformInitializerFilter.MOMENTS
        )
        init.SetAngle(np.deg2rad(angle_deg))
        tx, score = run(init, n_iter)
        if score < best_score:
            best_score, best_tx = score, tx

    refined_init = sitk.AffineTransform(2)
    refined_init.SetCenter(best_tx.GetCenter())
    refined_init.SetMatrix(best_tx.GetMatrix())
    refined_init.SetTranslation(best_tx.GetTranslation())

    final_tx, _ = run(refined_init, refine_iter)
    return final_tx

schedules = {
    "fine-only (no pyramid)": ([1], [0]),
    "2-level coarse-to-fine": ([2, 1], [1, 0]),
    "3-level coarse-to-fine (original)": ([4, 2, 1], [2, 1, 0]),
}

results = {}
for name, (shrink, sigma) in schedules.items():
    start = time.time()
    tx = register_mi_with_schedule(fixed_pil, moving_pil, shrink, sigma)
    elapsed = time.time() - start

    H_est = sitk_to_forward_matrix(tx)
    err = corner_error(H_true, H_est, size=fixed_pil.size[0])

    results[name] = (elapsed, err)
    print(f"{name:35s} | time: {elapsed:6.2f}s | corner error: {err:7.2f}px")

Multi-resolution pyramids accelerate convergence, but over-smoothing at very coarse levels ($4\times$ downsampling) can degrade accuracy by washing out critical high-frequency SAR features needed for precise metric evaluation. For this dataset, a 2-level coarse-to-fine strategy provides the optimal balance of runtime speed and registration accuracy.